In [157]:
from __future__ import annotations
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Tuple, Optional
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
import zipfile
import requests
import io

## Rishi's part
data cleaning and preprocessing

In [158]:
def load_electricity_data():
    url = "https://archive.ics.uci.edu/static/public/321/electricityloaddiagrams20112014.zip"
    response = requests.get(url)
    response.raise_for_status()

    z = zipfile.ZipFile(io.BytesIO(response.content))
    file_name = z.namelist()[0]

    df = pd.read_csv(
        z.open(file_name),
        sep=";",
        decimal=",",
        parse_dates=[0],
        quotechar='"'
    )

    df.rename(columns={df.columns[0]: "datetime"}, inplace=True)
    return df

In [159]:

def aggregate_data(df):
    df = df.copy()

    df["datetime"] = pd.to_datetime(df["datetime"])

    # Sum only customer load columns, not datetime
    load_cols = [col for col in df.columns if col != "datetime"]
    df["total_load"] = df[load_cols].sum(axis=1)

    # Resample using datetime, then bring it back as a column
    df_hourly = (
        df[["datetime", "total_load"]]
        .set_index("datetime")
        .resample("h")
        .sum()
        .reset_index()
    )

    return df_hourly

In [160]:
def create_features(df_hourly):
    df_features = df_hourly.copy()

    if "datetime" not in df_features.columns:
        raise ValueError("Expected a 'datetime' column in df_hourly.")

    df_features["datetime"] = pd.to_datetime(df_features["datetime"])

    df_features["hour_of_day"] = df_features["datetime"].dt.hour
    df_features["day_of_week"] = df_features["datetime"].dt.dayofweek
    df_features["is_weekend"] = (df_features["day_of_week"] >= 5).astype(int)
    df_features["month"] = df_features["datetime"].dt.month

    df_features["hour_sin"] = np.sin(2 * np.pi * df_features["hour_of_day"] / 24)
    df_features["hour_cos"] = np.cos(2 * np.pi * df_features["hour_of_day"] / 24)

    df_features["dayofweek_sin"] = np.sin(2 * np.pi * df_features["day_of_week"] / 7)
    df_features["dayofweek_cos"] = np.cos(2 * np.pi * df_features["day_of_week"] / 7)

    df_features["month_sin"] = np.sin(2 * np.pi * (df_features["month"] - 1) / 12)
    df_features["month_cos"] = np.cos(2 * np.pi * (df_features["month"] - 1) / 12)

    return df_features

In [161]:
df = load_electricity_data()
df_hourly = aggregate_data(df)
df = create_features(df_hourly)

In [162]:
df.columns

Index(['datetime', 'total_load', 'hour_of_day', 'day_of_week', 'is_weekend',
       'month', 'hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos',
       'month_sin', 'month_cos'],
      dtype='object')

## Baseline implementation

In [163]:
# ============================================================
# Config
# ============================================================

@dataclass
class Config:
    timestamp_col: str = "datetime"
    target_col: str = "total_load"

    input_window: int = 168
    forecast_horizon: int = 24

    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15

    batch_size: int = 64
    hidden_size: int = 64
    num_layers: int = 2
    dropout: float = 0.1
    learning_rate: float = 1e-3
    num_epochs: int = 20
    patience: int = 5

    seasonal_lag_day: int = 24
    seasonal_lag_week: int = 24 * 7

    train_shuffle: bool = True
    evaluate_test: bool = False
    output_dir: str = "person2_outputs"

    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


def validate_hourly_df(
    df: pd.DataFrame,
    timestamp_col: str,
    target_col: str,
) -> pd.DataFrame:
    df = df.copy()

    if timestamp_col not in df.columns:
        raise ValueError(f"Missing timestamp column: {timestamp_col}")
    if target_col not in df.columns:
        raise ValueError(f"Missing target column: {target_col}")

    df[timestamp_col] = pd.to_datetime(df[timestamp_col])
    df = df.sort_values(timestamp_col).reset_index(drop=True)

    if df[target_col].isna().any():
        raise ValueError(f"Target column '{target_col}' contains missing values.")

    if df[timestamp_col].duplicated().any():
        dup_count = int(df[timestamp_col].duplicated().sum())
        raise ValueError(f"Found {dup_count} duplicate timestamps in '{timestamp_col}'.")

    if not df[timestamp_col].is_monotonic_increasing:
        raise ValueError(f"Timestamp column '{timestamp_col}' is not strictly increasing.")

    time_diffs = df[timestamp_col].diff().dropna()
    if not time_diffs.empty:
        most_common_diff = time_diffs.mode().iloc[0]
        if most_common_diff != pd.Timedelta(hours=1):
            raise ValueError(
                f"Most common time delta is {most_common_diff}, not 1 hour. "
                "Make sure the dataframe is already hourly."
            )

        expected_range = pd.date_range(
            start=df[timestamp_col].min(),
            end=df[timestamp_col].max(),
            freq="h"
        )
        missing_timestamps = expected_range.difference(df[timestamp_col])

        if len(missing_timestamps) > 0:
            raise ValueError(
                f"Found {len(missing_timestamps)} missing hourly timestamps between "
                f"{df[timestamp_col].min()} and {df[timestamp_col].max()}."
            )

    return df
# Reproducibility

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)



In [164]:
def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mape(y_true: np.ndarray, y_pred: np.ndarray, eps: float = 1e-8) -> float:
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MAPE": mape(y_true, y_pred),
    }


def compute_horizon_metrics(y_true_2d: np.ndarray, y_pred_2d: np.ndarray) -> pd.DataFrame:
    """
    y_true_2d, y_pred_2d: shape [N, horizon]
    Returns one row per forecast horizon step.
    """
    rows = []
    horizon = y_true_2d.shape[1]

    for h in range(horizon):
        rows.append({
            "horizon_step": h + 1,
            "MAE": mae(y_true_2d[:, h], y_pred_2d[:, h]),
            "RMSE": rmse(y_true_2d[:, h], y_pred_2d[:, h]),
            "MAPE": mape(y_true_2d[:, h], y_pred_2d[:, h]),
        })

    return pd.DataFrame(rows)

In [165]:
def summarize_split(df: pd.DataFrame, timestamp_col: str, target_col: str, split_name: str) -> None:
    print(
        f"{split_name}: rows={len(df)}, "
        f"start={df[timestamp_col].min()}, "
        f"end={df[timestamp_col].max()}, "
        f"{target_col}_mean={df[target_col].mean():.2f}"
    )

In [166]:

# Data loading and splitting

def chronological_split(
    df: pd.DataFrame,
    train_ratio: float,
    val_ratio: float,
    test_ratio: float,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    total = len(df)
    if not math.isclose(train_ratio + val_ratio + test_ratio, 1.0, rel_tol=1e-6):
        raise ValueError("Train/val/test ratios must sum to 1.")

    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()

    return train_df, val_df, test_df



In [167]:
# Baselines

def make_targets_for_eval(
    series: np.ndarray,
    input_window: int,
    forecast_horizon: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Creates aligned windows for evaluation.
    Returns:
        X_context: shape [num_samples, input_window]
        y_true:    shape [num_samples, forecast_horizon]
    """
    X_context, y_true = [], []
    max_start = len(series) - input_window - forecast_horizon + 1

    for start in range(max_start):
        in_end = start + input_window
        out_end = in_end + forecast_horizon

        X_context.append(series[start:in_end])
        y_true.append(series[in_end:out_end])

    return np.asarray(X_context), np.asarray(y_true)


def naive_last_value_forecast(
    X_context: np.ndarray,
    forecast_horizon: int,
) -> np.ndarray:
    """
    Repeat the last observed value across the forecast horizon.
    """
    last_vals = X_context[:, -1]
    return np.repeat(last_vals[:, None], forecast_horizon, axis=1)


def seasonal_naive_forecast(
    X_context: np.ndarray,
    forecast_horizon: int,
    seasonal_lag: int,
) -> np.ndarray:
    """
    Seasonal naive forecast:
    for each forecast step h, copy the value from h steps ahead
    within the previous seasonal cycle.

    Example:
    - seasonal_lag=24  -> same hour previous day
    - seasonal_lag=168 -> same hour previous week
    """
    if X_context.shape[1] < seasonal_lag:
        raise ValueError(
            f"Input window must be at least seasonal_lag={seasonal_lag}, "
            f"but got {X_context.shape[1]}"
        )

    preds = []
    for h in range(forecast_horizon):
        idx = -seasonal_lag + h
        preds.append(X_context[:, idx])

    return np.stack(preds, axis=1)

In [168]:
# LSTM dataset

def build_supervised_sequences(
    series: np.ndarray,
    input_window: int,
    forecast_horizon: int,
) -> Tuple[np.ndarray, np.ndarray]:
    X, y = [], []
    max_start = len(series) - input_window - forecast_horizon + 1

    for start in range(max_start):
        in_end = start + input_window
        out_end = in_end + forecast_horizon

        X.append(series[start:in_end])
        y.append(series[in_end:out_end])

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    return X, y


class SequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray) -> None:
        self.X = torch.from_numpy(X).unsqueeze(-1)  # [N, seq_len, 1]
        self.y = torch.from_numpy(y)                # [N, horizon]

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]

# LSTM model

class LSTMForecaster(nn.Module):
    def __init__(
        self,
        input_size: int = 1,
        hidden_size: int = 64,
        num_layers: int = 2,
        dropout: float = 0.1,
        forecast_horizon: int = 24,
    ) -> None:
        super().__init__()

        lstm_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=lstm_dropout,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, forecast_horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, seq_len, 1]
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]       # [batch, hidden_size]
        last_hidden = self.dropout(last_hidden)
        preds = self.fc(last_hidden)      # [batch, horizon]
        return preds



In [169]:
# Train / eval loops

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: str,
) -> float:
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_loss(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: str,
) -> float:
    model.eval()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict_lstm(
    model: nn.Module,
    loader: DataLoader,
    device: str,
) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    preds_all, y_all = [], []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch).cpu().numpy()
        preds_all.append(preds)
        y_all.append(y_batch.numpy())

    return np.vstack(preds_all), np.vstack(y_all)


def inverse_transform_2d(
    scaler: StandardScaler,
    arr_2d: np.ndarray,
) -> np.ndarray:
    """
    Inverse transforms a [N, horizon] array using a scaler fit on a single feature.
    """
    flat = arr_2d.reshape(-1, 1)
    inv = scaler.inverse_transform(flat).reshape(arr_2d.shape)
    return inv



In [170]:
FEATURE_COLS = [
    "total_load",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dayofweek_sin",
    "dayofweek_cos",
    "month_sin",
    "month_cos",
]

In [171]:
def build_multivariate_sequences(
    feature_df: pd.DataFrame,
    target_array: np.ndarray,
    feature_cols: list[str],
    input_window: int,
    forecast_horizon: int,
) -> Tuple[np.ndarray, np.ndarray]:
    X, y = [], []

    feature_array = feature_df[feature_cols].to_numpy(dtype=np.float32)
    target_array = np.asarray(target_array, dtype=np.float32)

    max_start = len(feature_df) - input_window - forecast_horizon + 1

    for start in range(max_start):
        in_end = start + input_window
        out_end = in_end + forecast_horizon

        X.append(feature_array[start:in_end])
        y.append(target_array[in_end:out_end])

    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)

In [172]:
class MultiFeatureSequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray) -> None:
        self.X = torch.from_numpy(X)   # [N, seq_len, num_features]
        self.y = torch.from_numpy(y)   # [N, horizon]

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]

In [173]:
class MultiFeatureLSTMForecaster(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int = 64,
        num_layers: int = 2,
        dropout: float = 0.1,
        forecast_horizon: int = 24,
    ) -> None:
        super().__init__()

        lstm_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=lstm_dropout,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, forecast_horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, seq_len, num_features]
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]      # [batch, hidden_size]
        last_hidden = self.dropout(last_hidden)
        preds = self.fc(last_hidden)     # [batch, horizon]
        return preds

In [174]:
def scale_feature_splits(
    train_df: pd.DataFrame,
    val_context_df: pd.DataFrame,
    test_context_df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, StandardScaler, StandardScaler]:
    feature_scaler = StandardScaler()
    target_scaler = StandardScaler()

    train_scaled = train_df.copy()
    val_scaled = val_context_df.copy()
    test_scaled = test_context_df.copy()

    # Scale all input features using train-only fit
    train_scaled[feature_cols] = feature_scaler.fit_transform(train_df[feature_cols])
    val_scaled[feature_cols] = feature_scaler.transform(val_context_df[feature_cols])
    test_scaled[feature_cols] = feature_scaler.transform(test_context_df[feature_cols])

    # Separate target scaler for inverse-transforming predictions
    target_scaler.fit(train_df[[target_col]])

    return train_scaled, val_scaled, test_scaled, feature_scaler, target_scaler

In [175]:
def train_and_evaluate_lstm(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    test_loader: DataLoader,
    target_scaler: StandardScaler,
    cfg: Config,
    model_label: str,
) -> Tuple[dict, np.ndarray, np.ndarray, Optional[np.ndarray], Optional[np.ndarray], dict]:
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    print(f"\n===== Training {model_label} =====")
    for epoch in range(1, cfg.num_epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=cfg.device,
        )
        val_loss = evaluate_loss(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=cfg.device,
        )

        print(f"{model_label} | Epoch {epoch:02d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= cfg.patience:
            print(f"{model_label} | Early stopping triggered.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    val_pred_scaled, y_val_scaled = predict_lstm(model, val_loader, cfg.device)
    val_pred = inverse_transform_2d(target_scaler, val_pred_scaled)
    y_val_true = inverse_transform_2d(target_scaler, y_val_scaled)

    results = {
        f"{model_label}_Val": compute_metrics(
            y_val_true.reshape(-1), val_pred.reshape(-1)
        )
    }

    test_pred = None
    y_test_true = None

    if cfg.evaluate_test:
        test_pred_scaled, y_test_scaled = predict_lstm(model, test_loader, cfg.device)
        test_pred = inverse_transform_2d(target_scaler, test_pred_scaled)
        y_test_true = inverse_transform_2d(target_scaler, y_test_scaled)

        results[f"{model_label}_Test"] = compute_metrics(
            y_test_true.reshape(-1), test_pred.reshape(-1)
        )

    return results, y_val_true, val_pred, y_test_true, test_pred, best_state

In [176]:
def main(df: pd.DataFrame) -> None:
    cfg = Config()
    set_seed(cfg.seed)

    print(f"Using device: {cfg.device}")

    df = validate_hourly_df(
        df=df,
        timestamp_col=cfg.timestamp_col,
        target_col=cfg.target_col,
    )

    train_df, val_df, test_df = chronological_split(
        df,
        train_ratio=cfg.train_ratio,
        val_ratio=cfg.val_ratio,
        test_ratio=cfg.test_ratio,
    )

    summarize_split(train_df, cfg.timestamp_col, cfg.target_col, "TRAIN")
    summarize_split(val_df, cfg.timestamp_col, cfg.target_col, "VAL")
    summarize_split(test_df, cfg.timestamp_col, cfg.target_col, "TEST")

    train_series = train_df[cfg.target_col].to_numpy(dtype=np.float32)
    val_series = val_df[cfg.target_col].to_numpy(dtype=np.float32)
    test_series = test_df[cfg.target_col].to_numpy(dtype=np.float32)

    val_context_series = np.concatenate([
        train_series[-cfg.input_window:],
        val_series
    ])
    test_context_series = np.concatenate([
        val_series[-cfg.input_window:],
        test_series
    ])

    X_val_ctx, y_val_true = make_targets_for_eval(
        val_context_series, cfg.input_window, cfg.forecast_horizon
    )
    X_test_ctx, y_test_true = make_targets_for_eval(
        test_context_series, cfg.input_window, cfg.forecast_horizon
    )

    # =========================================================
    # Baselines
    # =========================================================
    val_pred_naive = naive_last_value_forecast(X_val_ctx, cfg.forecast_horizon)
    val_pred_seasonal_day = seasonal_naive_forecast(
        X_val_ctx, cfg.forecast_horizon, cfg.seasonal_lag_day
    )
    val_pred_seasonal_week = seasonal_naive_forecast(
        X_val_ctx, cfg.forecast_horizon, cfg.seasonal_lag_week
    )

    baseline_results = {
        "Naive_Last_Value_Val": compute_metrics(
            y_val_true.reshape(-1), val_pred_naive.reshape(-1)
        ),
        "Seasonal_Naive_Day_Val": compute_metrics(
            y_val_true.reshape(-1), val_pred_seasonal_day.reshape(-1)
        ),
        "Seasonal_Naive_Week_Val": compute_metrics(
            y_val_true.reshape(-1), val_pred_seasonal_week.reshape(-1)
        ),
    }

    test_pred_naive = None
    test_pred_seasonal_day = None
    test_pred_seasonal_week = None

    if cfg.evaluate_test:
        test_pred_naive = naive_last_value_forecast(X_test_ctx, cfg.forecast_horizon)
        test_pred_seasonal_day = seasonal_naive_forecast(
            X_test_ctx, cfg.forecast_horizon, cfg.seasonal_lag_day
        )
        test_pred_seasonal_week = seasonal_naive_forecast(
            X_test_ctx, cfg.forecast_horizon, cfg.seasonal_lag_week
        )

        baseline_results.update({
            "Naive_Last_Value_Test": compute_metrics(
                y_test_true.reshape(-1), test_pred_naive.reshape(-1)
            ),
            "Seasonal_Naive_Day_Test": compute_metrics(
                y_test_true.reshape(-1), test_pred_seasonal_day.reshape(-1)
            ),
            "Seasonal_Naive_Week_Test": compute_metrics(
                y_test_true.reshape(-1), test_pred_seasonal_week.reshape(-1)
            ),
        })

    print("\n===== Baseline Results =====")
    for name, metrics in baseline_results.items():
        print(name, metrics)

    # =========================================================
    # Shared context dataframes for multivariate model
    # =========================================================
    feature_cols = FEATURE_COLS
    missing_features = [col for col in feature_cols if col not in df.columns]
    if missing_features:
        raise ValueError(f"Missing required feature columns: {missing_features}")

    val_context_df = pd.concat(
        [train_df.tail(cfg.input_window), val_df],
        axis=0
    ).reset_index(drop=True)

    test_context_df = pd.concat(
        [val_df.tail(cfg.input_window), test_df],
        axis=0
    ).reset_index(drop=True)

    # =========================================================
    # Univariate LSTM
    # =========================================================
    scaler_uni = StandardScaler()
    train_scaled_uni = scaler_uni.fit_transform(train_series.reshape(-1, 1)).reshape(-1)
    val_scaled_full_uni = scaler_uni.transform(val_context_series.reshape(-1, 1)).reshape(-1)
    test_scaled_full_uni = scaler_uni.transform(test_context_series.reshape(-1, 1)).reshape(-1)

    X_train_uni, y_train_uni = build_supervised_sequences(
        train_scaled_uni, cfg.input_window, cfg.forecast_horizon
    )
    X_val_uni, y_val_uni = build_supervised_sequences(
        val_scaled_full_uni, cfg.input_window, cfg.forecast_horizon
    )
    X_test_uni, y_test_uni = build_supervised_sequences(
        test_scaled_full_uni, cfg.input_window, cfg.forecast_horizon
    )

    train_ds_uni = SequenceDataset(X_train_uni, y_train_uni)
    val_ds_uni = SequenceDataset(X_val_uni, y_val_uni)
    test_ds_uni = SequenceDataset(X_test_uni, y_test_uni)

    train_loader_uni = DataLoader(
        train_ds_uni, batch_size=cfg.batch_size, shuffle=cfg.train_shuffle
    )
    val_loader_uni = DataLoader(
        val_ds_uni, batch_size=cfg.batch_size, shuffle=False
    )
    test_loader_uni = DataLoader(
        test_ds_uni, batch_size=cfg.batch_size, shuffle=False
    )

    model_uni = LSTMForecaster(
        input_size=1,
        hidden_size=cfg.hidden_size,
        num_layers=cfg.num_layers,
        dropout=cfg.dropout,
        forecast_horizon=cfg.forecast_horizon,
    ).to(cfg.device)

    uni_results, y_val_lstm_uni_true, val_pred_lstm_uni, y_test_lstm_uni_true, test_pred_lstm_uni, best_state_uni = train_and_evaluate_lstm(
        model=model_uni,
        train_loader=train_loader_uni,
        val_loader=val_loader_uni,
        test_loader=test_loader_uni,
        target_scaler=scaler_uni,
        cfg=cfg,
        model_label="Univariate_LSTM",
    )

        # =========================================================
    # Multivariate LSTM
    # =========================================================
    train_scaled_multi_df, val_scaled_multi_df, test_scaled_multi_df, feature_scaler_multi, target_scaler_multi = scale_feature_splits(
        train_df=train_df,
        val_context_df=val_context_df,
        test_context_df=test_context_df,
        feature_cols=feature_cols,
        target_col=cfg.target_col,
    )

    raw_train_target = train_df[cfg.target_col].to_numpy(dtype=np.float32)
    raw_val_target = val_context_df[cfg.target_col].to_numpy(dtype=np.float32)
    raw_test_target = test_context_df[cfg.target_col].to_numpy(dtype=np.float32)

    X_train_multi, y_train_multi = build_multivariate_sequences(
        feature_df=train_scaled_multi_df,
        target_array=raw_train_target,
        feature_cols=feature_cols,
        input_window=cfg.input_window,
        forecast_horizon=cfg.forecast_horizon,
    )

    X_val_multi, y_val_multi = build_multivariate_sequences(
        feature_df=val_scaled_multi_df,
        target_array=raw_val_target,
        feature_cols=feature_cols,
        input_window=cfg.input_window,
        forecast_horizon=cfg.forecast_horizon,
    )

    X_test_multi, y_test_multi = build_multivariate_sequences(
        feature_df=test_scaled_multi_df,
        target_array=raw_test_target,
        feature_cols=feature_cols,
        input_window=cfg.input_window,
        forecast_horizon=cfg.forecast_horizon,
    )

    y_train_multi = target_scaler_multi.transform(
        y_train_multi.reshape(-1, 1)
    ).reshape(y_train_multi.shape).astype(np.float32)

    y_val_multi = target_scaler_multi.transform(
        y_val_multi.reshape(-1, 1)
    ).reshape(y_val_multi.shape).astype(np.float32)

    y_test_multi = target_scaler_multi.transform(
        y_test_multi.reshape(-1, 1)
    ).reshape(y_test_multi.shape).astype(np.float32)

    train_ds_multi = MultiFeatureSequenceDataset(X_train_multi, y_train_multi)
    val_ds_multi = MultiFeatureSequenceDataset(X_val_multi, y_val_multi)
    test_ds_multi = MultiFeatureSequenceDataset(X_test_multi, y_test_multi)

    train_loader_multi = DataLoader(
        train_ds_multi, batch_size=cfg.batch_size, shuffle=cfg.train_shuffle
    )
    val_loader_multi = DataLoader(
        val_ds_multi, batch_size=cfg.batch_size, shuffle=False
    )
    test_loader_multi = DataLoader(
        test_ds_multi, batch_size=cfg.batch_size, shuffle=False
    )

    model_multi = MultiFeatureLSTMForecaster(
        input_size=len(feature_cols),
        hidden_size=cfg.hidden_size,
        num_layers=cfg.num_layers,
        dropout=cfg.dropout,
        forecast_horizon=cfg.forecast_horizon,
    ).to(cfg.device)

    multi_results, y_val_lstm_multi_true, val_pred_lstm_multi, y_test_lstm_multi_true, test_pred_lstm_multi, best_state_multi = train_and_evaluate_lstm(
        model=model_multi,
        train_loader=train_loader_multi,
        val_loader=val_loader_multi,
        test_loader=test_loader_multi,
        target_scaler=target_scaler_multi,
        cfg=cfg,
        model_label="Multivariate_LSTM",
    )

    lstm_results = {}
    lstm_results.update(uni_results)
    lstm_results.update(multi_results)

    print("\n===== LSTM Results =====")
    for name, metrics in lstm_results.items():
        print(name, metrics)

    # =========================================================
    # Save outputs
    # =========================================================
    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    metrics_rows = []
    for model_name, metrics in {**baseline_results, **lstm_results}.items():
        row = {"model": model_name}
        row.update(metrics)
        metrics_rows.append(row)

    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df.to_csv(output_dir / "benchmark_metrics.csv", index=False)

    # Validation horizon-wise metrics
    compute_horizon_metrics(
        y_val_true, val_pred_naive
    ).to_csv(output_dir / "val_horizon_metrics_naive.csv", index=False)

    compute_horizon_metrics(
        y_val_true, val_pred_seasonal_day
    ).to_csv(output_dir / "val_horizon_metrics_seasonal_day.csv", index=False)

    compute_horizon_metrics(
        y_val_true, val_pred_seasonal_week
    ).to_csv(output_dir / "val_horizon_metrics_seasonal_week.csv", index=False)

    compute_horizon_metrics(
        y_val_lstm_uni_true, val_pred_lstm_uni
    ).to_csv(output_dir / "val_horizon_metrics_univariate_lstm.csv", index=False)

    compute_horizon_metrics(
        y_val_lstm_multi_true, val_pred_lstm_multi
    ).to_csv(output_dir / "val_horizon_metrics_multivariate_lstm.csv", index=False)

    # Validation predictions
    np.save(output_dir / "y_val_true_baselines.npy", y_val_true)
    np.save(output_dir / "val_pred_naive.npy", val_pred_naive)
    np.save(output_dir / "val_pred_seasonal_day.npy", val_pred_seasonal_day)
    np.save(output_dir / "val_pred_seasonal_week.npy", val_pred_seasonal_week)

    np.save(output_dir / "y_val_true_univariate_lstm.npy", y_val_lstm_uni_true)
    np.save(output_dir / "val_pred_univariate_lstm.npy", val_pred_lstm_uni)

    np.save(output_dir / "y_val_true_multivariate_lstm.npy", y_val_lstm_multi_true)
    np.save(output_dir / "val_pred_multivariate_lstm.npy", val_pred_lstm_multi)

    if cfg.evaluate_test:
        np.save(output_dir / "y_test_true_baselines.npy", y_test_true)
        np.save(output_dir / "test_pred_naive.npy", test_pred_naive)
        np.save(output_dir / "test_pred_seasonal_day.npy", test_pred_seasonal_day)
        np.save(output_dir / "test_pred_seasonal_week.npy", test_pred_seasonal_week)

        np.save(output_dir / "y_test_true_univariate_lstm.npy", y_test_lstm_uni_true)
        np.save(output_dir / "test_pred_univariate_lstm.npy", test_pred_lstm_uni)

        np.save(output_dir / "y_test_true_multivariate_lstm.npy", y_test_lstm_multi_true)
        np.save(output_dir / "test_pred_multivariate_lstm.npy", test_pred_lstm_multi)

        compute_horizon_metrics(
            y_test_true, test_pred_naive
        ).to_csv(output_dir / "test_horizon_metrics_naive.csv", index=False)

        compute_horizon_metrics(
            y_test_true, test_pred_seasonal_day
        ).to_csv(output_dir / "test_horizon_metrics_seasonal_day.csv", index=False)

        compute_horizon_metrics(
            y_test_true, test_pred_seasonal_week
        ).to_csv(output_dir / "test_horizon_metrics_seasonal_week.csv", index=False)

        compute_horizon_metrics(
            y_test_lstm_uni_true, test_pred_lstm_uni
        ).to_csv(output_dir / "test_horizon_metrics_univariate_lstm.csv", index=False)

        compute_horizon_metrics(
            y_test_lstm_multi_true, test_pred_lstm_multi
        ).to_csv(output_dir / "test_horizon_metrics_multivariate_lstm.csv", index=False)

    torch.save(best_state_uni, output_dir / "univariate_lstm_model.pt")
    torch.save(best_state_multi, output_dir / "multivariate_lstm_model.pt")

    metadata = {
        "timestamp_col": cfg.timestamp_col,
        "target_col": cfg.target_col,
        "input_window": cfg.input_window,
        "forecast_horizon": cfg.forecast_horizon,
        "train_ratio": cfg.train_ratio,
        "val_ratio": cfg.val_ratio,
        "test_ratio": cfg.test_ratio,
        "batch_size": cfg.batch_size,
        "hidden_size": cfg.hidden_size,
        "num_layers": cfg.num_layers,
        "dropout": cfg.dropout,
        "learning_rate": cfg.learning_rate,
        "num_epochs": cfg.num_epochs,
        "patience": cfg.patience,
        "seasonal_lag_day": cfg.seasonal_lag_day,
        "seasonal_lag_week": cfg.seasonal_lag_week,
        "train_shuffle": cfg.train_shuffle,
        "evaluate_test": cfg.evaluate_test,
        "seed": cfg.seed,
        "univariate_lstm_input_type": "total_load_only",
        "multivariate_lstm_input_type": "engineered_features",
        "multivariate_feature_columns": feature_cols,
        "train_start": str(train_df[cfg.timestamp_col].min()),
        "train_end": str(train_df[cfg.timestamp_col].max()),
        "val_start": str(val_df[cfg.timestamp_col].min()),
        "val_end": str(val_df[cfg.timestamp_col].max()),
        "test_start": str(test_df[cfg.timestamp_col].min()),
        "test_end": str(test_df[cfg.timestamp_col].max()),
        "univariate_scaler_mean": float(scaler_uni.mean_[0]),
        "univariate_scaler_scale": float(scaler_uni.scale_[0]),
        "multivariate_target_scaler_mean": float(target_scaler_multi.mean_[0]),
        "multivariate_target_scaler_scale": float(target_scaler_multi.scale_[0]),
    }

    pd.Series(metadata).to_json(output_dir / "run_metadata.json", indent=2)

    print(f"\nSaved outputs to: {output_dir.resolve()}")

In [177]:
main(df)

Using device: cuda
TRAIN: rows=24545, start=2011-01-01 00:00:00, end=2013-10-19 16:00:00, total_load_mean=738475.39
VAL: rows=5259, start=2013-10-19 17:00:00, end=2014-05-26 19:00:00, total_load_mean=806795.72
TEST: rows=5261, start=2014-05-26 20:00:00, end=2015-01-01 00:00:00, total_load_mean=961645.58

===== Baseline Results =====
Naive_Last_Value_Val {'MAE': 255663.390625, 'RMSE': 327192.375, 'MAPE': 44.57447052001953}
Seasonal_Naive_Day_Val {'MAE': 28049.154296875, 'RMSE': 57315.078125, 'MAPE': 9.243870735168457}
Seasonal_Naive_Week_Val {'MAE': 35390.171875, 'RMSE': 63164.265625, 'MAPE': 9.629261016845703}

===== Training Univariate_LSTM =====
Univariate_LSTM | Epoch 01 | Train Loss: 0.183404 | Val Loss: 0.033227
Univariate_LSTM | Epoch 02 | Train Loss: 0.037266 | Val Loss: 0.027012
Univariate_LSTM | Epoch 03 | Train Loss: 0.032014 | Val Loss: 0.025286
Univariate_LSTM | Epoch 04 | Train Loss: 0.029613 | Val Loss: 0.024207
Univariate_LSTM | Epoch 05 | Train Loss: 0.028333 | Val Loss

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



===== Training Multivariate_LSTM =====
Multivariate_LSTM | Epoch 01 | Train Loss: 0.132013 | Val Loss: 0.027939
Multivariate_LSTM | Epoch 02 | Train Loss: 0.032147 | Val Loss: 0.024938
Multivariate_LSTM | Epoch 03 | Train Loss: 0.028695 | Val Loss: 0.025487
Multivariate_LSTM | Epoch 04 | Train Loss: 0.026511 | Val Loss: 0.023428
Multivariate_LSTM | Epoch 05 | Train Loss: 0.025141 | Val Loss: 0.021690
Multivariate_LSTM | Epoch 06 | Train Loss: 0.024379 | Val Loss: 0.022812
Multivariate_LSTM | Epoch 07 | Train Loss: 0.023509 | Val Loss: 0.018565
Multivariate_LSTM | Epoch 08 | Train Loss: 0.023174 | Val Loss: 0.021026
Multivariate_LSTM | Epoch 09 | Train Loss: 0.022721 | Val Loss: 0.022319
Multivariate_LSTM | Epoch 10 | Train Loss: 0.022058 | Val Loss: 0.017675
Multivariate_LSTM | Epoch 11 | Train Loss: 0.021554 | Val Loss: 0.018174
Multivariate_LSTM | Epoch 12 | Train Loss: 0.021090 | Val Loss: 0.018690
Multivariate_LSTM | Epoch 13 | Train Loss: 0.020774 | Val Loss: 0.019610
Multivariat